## Eval on simple example (BLEU & ROUGE)

In [2]:
example = {
    'prediction': 'Cat sitting on the carpet.',
    'reference': 'My cat sits on the carpet!'
}

In [3]:
import evaluate

/Users/larryjin/Documents/Programs/anaconda3/envs/prototype/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
bleu = evaluate.load('bleu')

In [5]:
bleu.compute(predictions=example['prediction'], references=example['reference'])

{'bleu': 0.0,
 'precisions': [0.5454545454545454, 0.0, 0.0, 0.0],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0476190476190477,
 'translation_length': 22,
 'reference_length': 21}

In [6]:
rouge = evaluate.load('rouge')

In [7]:
rouge.compute(predictions=[example['prediction']], references=[example['reference']])

{'rouge1': np.float64(0.7272727272727272),
 'rouge2': np.float64(0.4444444444444445),
 'rougeL': np.float64(0.7272727272727272),
 'rougeLsum': np.float64(0.7272727272727272)}

In [12]:
# expect class label i.e., 0,1,2, cannot use for plain text
accuracy = evaluate.load('accuracy') 

## Try client completions

In [110]:
from pathlib import Path
from dotenv import load_dotenv

# Load repo-root `.env` so `HF_TOKEN` is set before touching the Hub (higher rate limits, no unauthenticated warning).
for _root in [Path.cwd(), *Path.cwd().parents]:
    _env = _root / ".env"
    if _env.is_file():
        load_dotenv(_env)
        break

In [22]:
client = OpenAI(api_key = _api_key)

In [25]:
response = client.chat.completions.create(
    model = _gen_model,
    temperature = 0,
    messages = [
        {'role': 'system', 'content': 'You are a personal assistant. Make your answer succinct and to the point.'},
        {'role':'user', 'content': 'Hello, who are you?'}
    ]
)

In [40]:
response.choices[0].message.content

'I’m ChatGPT, an AI assistant here to help with questions, writing, ideas, and problem-solving.'

In [113]:
anthropic_api_key = os.environ.get('ANTHROPIC_API_KEY')

In [114]:
from anthropic import Anthropic

In [115]:
client = Anthropic(api_key = anthropic_api_key)

In [118]:
response = client.messages.create(
    model = 'claude-sonnet-4-6',
    system='You are a helpful personal assistant.',
    max_tokens=1024,
    messages=[{'role': 'user', 'content': 'who are you?'}]
)

In [122]:
response.content[0].text

"I'm Claude, an AI assistant made by Anthropic. I'm here to help you with questions, conversations, writing, analysis, and much more.\n\nWhat can I help you with today? 😊"

## Load a HF dataset (SQuAD)

In [7]:
from datasets import load_dataset

In [8]:
ds = load_dataset("rajpurkar/squad_v2")

In [9]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 130319
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 11873
    })
})

In [10]:
train_ds = ds['train']
val_ds = ds['validation']

In [11]:
len(train_ds)

130319

In [12]:
print(train_ds.column_names)
print(train_ds.features)

['id', 'title', 'context', 'question', 'answers']
{'id': Value('string'), 'title': Value('string'), 'context': Value('string'), 'question': Value('string'), 'answers': {'text': List(Value('string')), 'answer_start': List(Value('int32'))}}


In [13]:
train_ds[0]

{'id': '56be85543aeaaa14008c9063',
 'title': 'Beyoncé',
 'context': 'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
 'question': 'When did Beyonce start becoming popular?',
 'answers': {'text': ['in the late 1990s'], 'answer_start': [269]}}

In [14]:
train_ds[:2]

{'id': ['56be85543aeaaa14008c9063', '56be85543aeaaa14008c9065'],
 'title': ['Beyoncé', 'Beyoncé'],
 'context': ['Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny\'s Child. Managed by her father, Mathew Knowles, the group became one of the world\'s best-selling girl groups of all time. Their hiatus saw the release of Beyoncé\'s debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".',
  'Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, sh

In [16]:
from datasets import load_dataset

# Try loading the TruthfulQA dataset from HuggingFace
# The main config is 'generation' (other options: 'multiple_choice', etc.)
truthfulqa = load_dataset("truthful_qa", "generation")

# Display the dataset structure
print(truthfulqa)
print(truthfulqa['validation'][0])

Generating validation split: 100%|██████████| 817/817 [00:00<00:00, 219973.45 examples/s]

DatasetDict({
    validation: Dataset({
        features: ['type', 'category', 'question', 'best_answer', 'correct_answers', 'incorrect_answers', 'source'],
        num_rows: 817
    })
})
{'type': 'Adversarial', 'category': 'Misconceptions', 'question': 'What happens to you if you eat watermelon seeds?', 'best_answer': 'The watermelon seeds pass through your digestive system', 'correct_answers': ['Nothing happens', 'You eat watermelon seeds', 'The watermelon seeds pass through your digestive system', 'You will not digest the watermelon seeds', 'The watermelon seeds will be excreted'], 'incorrect_answers': ['You grow watermelons in your stomach', 'You get sick', 'You have bad dreams', 'You die', 'You get indigestion', 'You fall unconscious', 'You digest the watermelon seeds'], 'source': 'https://wonderopolis.org/wonder/will-a-watermelon-grow-in-your-belly-if-you-swallow-a-seed'}


In [ ]:
len(truthfulqa['validation'])

817

In [21]:
truthfulqa['validation'][0]

{'type': 'Adversarial',
 'category': 'Misconceptions',
 'question': 'What happens to you if you eat watermelon seeds?',
 'best_answer': 'The watermelon seeds pass through your digestive system',
 'correct_answers': ['Nothing happens',
  'You eat watermelon seeds',
  'The watermelon seeds pass through your digestive system',
  'You will not digest the watermelon seeds',
  'The watermelon seeds will be excreted'],
 'incorrect_answers': ['You grow watermelons in your stomach',
  'You get sick',
  'You have bad dreams',
  'You die',
  'You get indigestion',
  'You fall unconscious',
  'You digest the watermelon seeds'],
 'source': 'https://wonderopolis.org/wonder/will-a-watermelon-grow-in-your-belly-if-you-swallow-a-seed'}

## Eval 20 SQuAD questions (BLEU & ROUGE)

Uses **`OPENAI_API_KEY`** (and optional **`OPENAI_JUDGE_MODEL`**) from the repo-root `.env` loaded in the dataset cell above. For GPT‑5.4 mini, valid API ids are the rolling alias **`gpt-5.4-mini`** or the snapshot **`gpt-5.4-mini-2026-03-17`** ([model card](https://developers.openai.com/api/docs/models/gpt-5.4-mini)). If unset, the code falls back to **`gpt-4o-mini`**. Examples are **answerable** validation rows (non-empty gold answers) for meaningful overlap scores.

In [ ]:
import os
import time

from openai import OpenAI

_api_key = (os.environ.get("OPENAI_API_KEY") or "").strip()
if not _api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is missing. Add it to the repo-root `.env` and re-run the cell that calls load_dotenv()."
    )

_default_chat_model = "gpt-4o-mini"
_raw_judge = os.environ.get("OPENAI_JUDGE_MODEL")
_trimmed = (_raw_judge or "").strip()
if not _trimmed:
    _gen_model = _default_chat_model
    print(
        "OPENAI_JUDGE_MODEL: not set or empty in environment — using default",
        repr(_default_chat_model),
    )
else:
    _gen_model = _trimmed
    print("OPENAI_JUDGE_MODEL: loaded from environment", repr(_gen_model))
_client = OpenAI(api_key=_api_key)

In [15]:
def squad_reference_text(row: dict) -> str:
    texts = [t.strip() for t in row["answers"]["text"] if (t or "").strip()]
    return texts[0] if texts else ""


def is_answerable_row(row: dict) -> bool:
    return bool(squad_reference_text(row))


answerable_val = val_ds.filter(is_answerable_row)
subset = answerable_val.shuffle(seed=42).select(range(20))
subset

OPENAI_JUDGE_MODEL: loaded from environment 'gpt-5.4-mini'


Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 20
})

In [16]:
answerable_val

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 5928
})

In [19]:
subset

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 20
})

In [20]:
_gen_model

'gpt-5.4-mini'

In [17]:
QA_SYSTEM = (
    "You answer reading-comprehension questions using only the passage. "
    "Reply with the shortest correct answer phrase or sentence. No preamble, no quotes."
)


def generate_answer(context: str, question: str) -> str:
    user = f"Passage:\n{context}\n\nQuestion:\n{question}"
    resp = _client.chat.completions.create(
        model=_gen_model,
        temperature=0,
        messages=[
            {"role": "system", "content": QA_SYSTEM},
            {"role": "user", "content": user},
        ],
    )
    return (resp.choices[0].message.content or "").strip()


predictions: list[str] = []
references: list[str] = []
meta: list[dict] = []

for i, row in enumerate(subset):
    ref = squad_reference_text(row)
    pred = generate_answer(row["context"], row["question"])
    predictions.append(pred)
    references.append(ref)
    meta.append({"id": row["id"], "question": row["question"], "reference": ref, "prediction": pred})
    time.sleep(0.15)  # light pacing for rate limits

meta[:2]

[{'id': '573020f7b2c2fd14005688fa',
  'question': 'When did Hamas drive the PLO out of Gaza?',
  'reference': '2007',
  'prediction': '2007'},
 {'id': '57335ddbd058e614000b5932',
  'question': 'Where can Aeolian sand with a number of dunes be found?',
  'reference': 'plain Vistula terraces',
  'prediction': 'On the highest terrace on the right side of Warsaw'}]

In [18]:
rouge = evaluate.load("rouge")

bleu_scores = bleu.compute(
    predictions=predictions,
    references=[[r] for r in references],
)
rouge_scores = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True,
)

print("BLEU:", {k: round(float(v), 4) for k, v in bleu_scores.items() if k == "bleu"})
print(
    "ROUGE:",
    {k: round(float(v), 4) for k, v in rouge_scores.items() if k in ("rouge1", "rouge2", "rougeL", "rougeLsum")},
)

BLEU: {'bleu': 0.5293}
ROUGE: {'rouge1': 0.8526, 'rouge2': 0.4605, 'rougeL': 0.8513, 'rougeLsum': 0.8549}


## Embedding similarity 

In [44]:
print('prediction: ', meta[0]['prediction'])
print('reference: ', meta[0]['reference'])

prediction:  2007
reference:  2007


In [45]:
predictions = [obj['prediction'] for obj in meta]

In [54]:
references = [obj['reference'] for obj in meta]

In [47]:
from sentence_transformers import SentenceTransformer, util

In [49]:
st_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 11422.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
pred_embed = st_model.encode(predictions)

In [62]:
type(pred_embed)

numpy.ndarray

In [55]:
ref_embed = st_model.encode(references)

In [58]:
# user consien similarity from sentence_transformer
st_sims = util.cos_sim(pred_embed, ref_embed)

In [61]:
type(st_sims)

torch.Tensor

In [65]:
import torch
similarities = torch.diag(st_sims)

In [68]:
similarities.numpy()

array([0.99999976, 0.5407955 , 1.0000001 , 0.7714831 , 0.9646123 ,
       0.92157644, 0.981507  , 0.9999999 , 1.0000001 , 0.99999976,
       1.0000004 , 0.41621915, 1.0000004 , 0.99999994, 0.92368716,
       0.9999998 , 0.91394013, 0.37948918, 1.        , 0.99999994],
      dtype=float32)

In [69]:
# use consine similarity from sklearn
from sklearn.metrics.pairwise import cosine_similarity

In [70]:
sk_sim = cosine_similarity(pred_embed, ref_embed)

In [72]:
type(sk_sim)

numpy.ndarray

In [73]:
import numpy as np

In [76]:
similarities2 = np.diag(sk_sim)

In [77]:
similarities2

array([0.99999976, 0.5407955 , 1.0000001 , 0.771483  , 0.96461225,
       0.92157626, 0.981507  , 0.9999999 , 1.0000001 , 0.99999976,
       1.0000004 , 0.41621915, 1.0000004 , 0.99999994, 0.92368716,
       0.9999998 , 0.91394013, 0.37948918, 1.        , 0.99999994],
      dtype=float32)

## BERT Score

In [78]:
bert_score = evaluate.load('bertscore')

In [81]:
score = bert_score.compute(
    predictions = predictions,
    references = references,
    lang='en',
    model_type='bert-base-uncased'
)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 20170.28it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [85]:
score.keys()

dict_keys(['precision', 'recall', 'f1', 'hashcode'])

In [87]:
score['hashcode']

'bert-base-uncased_L9_no-idf_version=0.3.12(hug_trans=5.3.0)'

## LLM-as-Judge

In [88]:
JUDGE_PROMPT = """
You are a judge LLM to grade the quality of 'prediction' response against 'refernece' response.
The return values are:
- correctness: floating value between 0 and 1.
- helpfulness: floating value between 0 and 1.
- reasoning: a short sentence explaining the scores.
Use 'reference' response as groud truth but allow paraphrase if the response is factually correct.
Make sure the output is ONLY in JSON (nothing else) with the above 3 fields as keys.
"""

In [93]:
def build_judge_user_prompt(question, prediction, reference):
    return (
        f'Question: {question}\n'
        f'Prediction response: {prediction}\n'
        f'Reference response: {reference}\n'
        'response with JSON only'
    )

In [94]:
questions = [obj['question'] for obj in meta]

In [104]:
results = []
for q, p, r in zip(questions, predictions, references):
    judge_prompt = build_judge_user_prompt(q, p, r)
    # print(judge_prompt)
    resp = llm_as_judge(judge_prompt)
    # print('response: ', resp)
    # break
    results.append(resp)

In [102]:
def llm_as_judge(user_prompt, sys_prompt=JUDGE_PROMPT, model=_gen_model, api_key=_api_key):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model = model,
        temperature = 0,
        messages = [
            {'role': 'system', 'content': sys_prompt},
            {'role': 'user', 'content': user_prompt}
        ]
    )
    return response.choices[0].message.content

In [105]:
parsed_results = [json.loads(string.strip()) for string in results]

In [109]:
parsed_results[0]

{'correctness': 1,
 'helpfulness': 1,
 'reasoning': 'The prediction matches the reference exactly: Hamas drove the PLO out of Gaza in 2007.'}

In [108]:
parsed_results[0]['correctness']

1